In [1]:
import os
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor, Pool

from helpers.db_query import query_db

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DB_PATH = Path(
    os.environ.get(
        "FF_DB_PATH",
        r"C:\Users\terre\OneDrive\Desktop\Projects\Fantasy_Football_Database\data\fantasy_football.db",
    )
)
assert DB_PATH.exists(), f"DB not found: {DB_PATH}"
print(DB_PATH, f"{DB_PATH.stat().st_size / 1e6:.1f} MB")

C:\Users\terre\OneDrive\Desktop\Projects\Fantasy_Football_Database\data\fantasy_football.db 38.1 MB


In [2]:
games_query = """
SELECT *
FROM v_player_games
WHERE season_type = 2
  AND season BETWEEN 2021 AND 2025
ORDER BY athlete_id, season, week
"""
df = query_db(games_query)
df["game_date"] = pd.to_datetime(df["game_date"])
df = df.drop(columns=["raw_stats", "loaded_at"])
df


,display_name,position_abbr,athlete_id,event_id,season,season_type,week,game_date,team_id,team_abbr,opponent_id,opponent_abbr,home_away,result,team_score,opponent_score,is_all_star,fp_standard,fp_half_ppr,fp_ppr,fumbles,fumblesForced,fumblesLost,kicksBlocked,longReception,longRushing,receivingTargets,receivingTouchdowns,receivingYards,receptions,rushingAttempts,rushingTouchdowns,rushingYards,yardsPerReception,yardsPerRushAttempt,QBRating,adjQBR,completionPct,completions,interceptions,longPassing,passingAttempts,passingTouchdowns,passingYards,sacks,yardsPerPassAttempt
0,Joe Flacco,QB,11252,401326469,2021,2,10,2021-11-14,20,NYJ,2,BUF,home,L,17.0,45.0,0,5.88,5.88,5.88,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,158.3,100.0,100.0,3.0,0.0,25.0,3.0,1.0,47.0,0.0,15.7
1,Joe Flacco,QB,11252,401326486,2021,2,11,2021-11-21,20,NYJ,15,MIA,home,L,17.0,24.0,0,19.94,19.94,19.94,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,2.0,0.0,3.0,NaN,1.5,101.5,45.7,61.5,24.0,0.0,62.0,39.0,2.0,291.0,2.0,7.5
2,Joe Flacco,QB,11252,401437632,2022,2,1,2022-09-11,20,NYJ,33,BAL,home,L,9.0,24.0,0,14.36,14.36,14.36,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,1.0,0.0,0.0,NaN,0.0,74.8,18.3,62.7,37.0,1.0,24.0,59.0,1.0,309.0,3.0,5.2
3,Joe Flacco,QB,11252,401437635,2022,2,2,2022-09-18,20,NYJ,5,CLE,away,W,31.0,30.0,0,28.88,28.88,28.88,NaN,NaN,NaN,NaN,NaN,7.0,NaN,NaN,NaN,NaN,2.0,0.0,6.0,NaN,3.0,110.7,72.2,59.1,26.0,0.0,66.0,44.0,4.0,307.0,2.0,7.0
4,Joe Flacco,QB,11252,401437740,2022,2,3,2022-09-25,20,NYJ,4,CIN,home,L,12.0,27.0,0,7.40,7.40,7.40,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,53.8,18.2,53.9,28.0,2.0,23.0,52.0,0.0,285.0,4.0,5.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13477,Aaron Rodgers,QB,8439,401772791,2025,2,14,2025-12-07,23,PIT,33,BAL,away,W,27.0,22.0,0,21.36,21.36,21.36,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,2.0,1.0,0.0,NaN,0.0,103.1,59.0,67.7,23.0,0.0,52.0,34.0,1.0,284.0,0.0,8.4
13478,Aaron Rodgers,QB,8439,401772823,2025,2,15,2025-12-16,23,PIT,15,MIA,home,W,28.0,15.0,0,16.96,16.96,16.96,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,125.9,56.2,85.2,23.0,0.0,28.0,27.0,2.0,224.0,3.0,8.3
13479,Aaron Rodgers,QB,8439,401772806,2025,2,16,2025-12-21,23,PIT,8,DET,away,W,29.0,24.0,0,15.54,15.54,15.54,NaN,NaN,NaN,NaN,NaN,7.0,NaN,NaN,NaN,NaN,2.0,0.0,9.0,NaN,4.5,92.1,56.9,65.9,27.0,0.0,45.0,41.0,1.0,266.0,2.0,6.5
13480,Aaron Rodgers,QB,8439,401772807,2025,2,17,2025-12-28,23,PIT,5,CLE,away,L,6.0,13.0,0,7.32,7.32,7.32,NaN,NaN,NaN,NaN,NaN,6.0,NaN,NaN,NaN,NaN,1.0,0.0,6.0,NaN,6.0,64.9,18.3,53.9,21.0,0.0,29.0,39.0,0.0,168.0,2.0,4.3


In [3]:
matchups_query = """
select * from v_team_schedule
where season in (2021, 2022, 2023, 2024, 2025)
"""

df_matchups = query_db(matchups_query)

In [4]:
defense_query = """
select * from v_team_defense_seasons
where season in (2021, 2022, 2023, 2024, 2025)
"""

df_defense = query_db(defense_query)

In [5]:
# --- Player features: prior-season averages, home/away, days rest ---

ID_COLS = [
    "display_name", "position_abbr", "athlete_id", "event_id", "team_id", "team_abbr",
    "opponent_id", "opponent_abbr", "home_away", "result", "game_date",
]
CALENDAR_COLS = ["season", "season_type", "week", "is_all_star"]

# Alternate scorings of the target — near-duplicates of fp_ppr, so they add nothing.
SKIP_STATS = ["fp_standard", "fp_half_ppr"]

# Everything else is a per-game stat to average over the prior season.
STAT_COLS = [c for c in df.columns if c not in ID_COLS + CALENDAR_COLS + SKIP_STATS]

# Per-player season averages, then shift the season forward so each row joins to
# the season *before* it. Merging on season (rather than shifting rows) means a
# player who missed a year gets no carry-over from two years back.
season_avg = (
    df.groupby(["athlete_id", "season"])
    .agg(games=("event_id", "size"), **{c: (c, "mean") for c in STAT_COLS})
    .reset_index()
)
season_avg["season"] += 1
season_avg = season_avg.rename(
    columns={**{c: f"prev_{c}_avg" for c in STAT_COLS}, "games": "prev_games"}
)

feat = df.merge(season_avg, on=["athlete_id", "season"], how="left")

# 2021 has no prior season in this pull, so it drops out.
feat = feat[feat["season"] >= 2022].sort_values(["athlete_id", "season", "week"])

# Days since that player's previous game, reset each season.
# First appearance of a season has no predecessor -> standard 7 days.
feat["days_rest"] = feat.groupby(["athlete_id", "season"])["game_date"].diff().dt.days
feat["days_rest"] = feat["days_rest"].fillna(7).astype(int)

# team_id / opponent_id are numeric-looking strings; as categories CatBoost treats
# them as labels instead of ordered numbers.
CAT_COLS = ["home_away", "position_abbr", "team_id", "opponent_id"]
feat[CAT_COLS] = feat[CAT_COLS].astype("category")

PREV_COLS = [c for c in feat.columns if c.startswith("prev_")]
feat[PREV_COLS] = feat[PREV_COLS].fillna(0)

print(f"{feat.shape[0]:,} rows | {len(PREV_COLS)} prior-season features | seasons {sorted(feat['season'].unique())}")
for c in CAT_COLS:
    print(f"  {c}: {len(feat[c].cat.categories)} levels")
feat[["display_name", "position_abbr", "season", "week", "game_date", "home_away",
      "days_rest", "fp_ppr", "prev_fp_ppr_avg", "prev_games"]].head(10)

12,325 rows | 30 prior-season features | seasons [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  home_away: 2 levels
  position_abbr: 4 levels
  team_id: 32 levels
  opponent_id: 32 levels


,display_name,position_abbr,season,week,game_date,home_away,days_rest,fp_ppr,prev_fp_ppr_avg,prev_games
2,Joe Flacco,QB,2022,1,2022-09-11,home,7,14.36,12.910,2.0
3,Joe Flacco,QB,2022,2,2022-09-18,away,7,28.88,12.910,2.0
4,Joe Flacco,QB,2022,3,2022-09-25,home,7,7.40,12.910,2.0
5,Joe Flacco,QB,2022,14,2022-12-11,away,77,0.04,12.910,2.0
6,Joe Flacco,QB,2022,18,2023-01-08,away,28,5.96,12.910,2.0
7,Joe Flacco,QB,2023,13,2023-12-03,away,7,16.16,11.328,5.0
8,Joe Flacco,QB,2023,14,2023-12-10,home,7,22.34,11.328,5.0
9,Joe Flacco,QB,2023,15,2023-12-17,home,7,16.86,11.328,5.0
10,Joe Flacco,QB,2023,16,2023-12-24,away,7,22.82,11.328,5.0
11,Joe Flacco,QB,2023,17,2023-12-29,home,5,22.66,11.328,5.0


In [6]:
# --- Opponent defense: prior-season team stats ---
# Same shift-the-season trick as the player features. At prediction time the current
# season's defensive numbers don't exist yet, so last year's are the best available.

# "games" is 17 for nearly every team, so it carries no signal.
DEF_ID_COLS = ["team_id", "team_abbr", "team_name", "season", "games"]
DEF_STAT_COLS = [c for c in df_defense.columns if c not in DEF_ID_COLS]

opp_prev = df_defense[["team_id", "season"] + DEF_STAT_COLS].copy()
opp_prev["season"] += 1
opp_prev = opp_prev.rename(
    columns={**{c: f"opp_prev_{c}" for c in DEF_STAT_COLS}, "team_id": "opponent_id"}
)
# Match the key dtype so the merge doesn't downgrade opponent_id out of category.
opp_prev["opponent_id"] = opp_prev["opponent_id"].astype(feat["opponent_id"].dtype)

# Idempotent: drop any columns from a previous run of this cell before re-merging.
feat = feat.drop(columns=[c for c in feat.columns if c.startswith("opp_prev_")])
feat = feat.merge(opp_prev, on=["opponent_id", "season"], how="left")

OPP_COLS = [c for c in feat.columns if c.startswith("opp_prev_")]
feat[OPP_COLS] = feat[OPP_COLS].fillna(0)

unmatched = (feat[OPP_COLS] == 0).all(axis=1).sum()
print(f"{len(OPP_COLS)} opponent-defense features | {len(feat):,} rows | unmatched: {unmatched}")
feat[["display_name", "season", "week", "opponent_id",
      "opp_prev_points_allowed_pg", "opp_prev_pass_yards_allowed_pg",
      "opp_prev_rush_yards_allowed_pg", "opp_prev_sacks_pg"]].head()

24 opponent-defense features | 12,325 rows | unmatched: 0


,display_name,season,week,opponent_id,opp_prev_points_allowed_pg,opp_prev_pass_yards_allowed_pg,opp_prev_rush_yards_allowed_pg,opp_prev_sacks_pg
0,Joe Flacco,2022,1,33,23.06,278.94,84.47,2.00
1,Joe Flacco,2022,2,5,21.82,202.29,109.24,2.53
2,Joe Flacco,2022,3,4,22.12,248.35,102.47,2.47
3,Joe Flacco,2022,14,2,17.00,163.00,109.76,2.47
4,Joe Flacco,2022,18,15,21.94,227.71,109.82,2.82


In [7]:
# --- Modeling dataset ---
# Every same-game box-score column is dropped here: those describe the game being
# predicted, so keeping them would leak the target. Only prior-season averages and
# pre-kickoff context survive.

TARGET = "fp_ppr"

# Carried for slicing/inspection, not fed to the model.
META_COLS = ["display_name", "athlete_id", "season", "week"]

FEATURE_COLS = CAT_COLS + ["days_rest"] + PREV_COLS + OPP_COLS

model_df = feat[META_COLS + FEATURE_COLS + [TARGET]].reset_index(drop=True)

print(f"{model_df.shape[0]:,} rows x {model_df.shape[1]} cols "
      f"({len(FEATURE_COLS)} features + {len(META_COLS)} meta + target)")
print(f"  categorical      : {len(CAT_COLS)}  {CAT_COLS}")
print(f"  player prior-year: {len(PREV_COLS)}")
print(f"  opponent defense : {len(OPP_COLS)}")
print("NaNs:", int(model_df.isna().sum().sum()))
model_df.head()

12,325 rows x 64 cols (59 features + 4 meta + target)
  categorical      : 4  ['home_away', 'position_abbr', 'team_id', 'opponent_id']
  player prior-year: 30
  opponent defense : 24
NaNs: 0


,display_name,athlete_id,season,week,home_away,position_abbr,team_id,opponent_id,days_rest,prev_games,prev_team_score_avg,prev_opponent_score_avg,prev_fp_ppr_avg,prev_fumbles_avg,prev_fumblesForced_avg,prev_fumblesLost_avg,prev_kicksBlocked_avg,prev_longReception_avg,prev_longRushing_avg,prev_receivingTargets_avg,prev_receivingTouchdowns_avg,prev_receivingYards_avg,prev_receptions_avg,prev_rushingAttempts_avg,prev_rushingTouchdowns_avg,prev_rushingYards_avg,prev_yardsPerReception_avg,prev_yardsPerRushAttempt_avg,prev_QBRating_avg,prev_adjQBR_avg,prev_completionPct_avg,prev_completions_avg,prev_interceptions_avg,prev_longPassing_avg,prev_passingAttempts_avg,prev_passingTouchdowns_avg,prev_passingYards_avg,prev_sacks_avg,prev_yardsPerPassAttempt_avg,opp_prev_points_allowed_pg,opp_prev_yards_allowed_pg,opp_prev_pass_yards_allowed_pg,opp_prev_rush_yards_allowed_pg,opp_prev_first_downs_allowed_pg,opp_prev_pass_attempts_allowed_pg,opp_prev_completions_allowed_pg,opp_prev_rush_attempts_allowed_pg,opp_prev_pass_tds_allowed_pg,opp_prev_rush_tds_allowed_pg,opp_prev_tds_allowed_pg,opp_prev_possession_seconds_allowed_pg,opp_prev_turnovers_forced_pg,opp_prev_takeaways_pg,opp_prev_fumbles_recovered_pg,opp_prev_sacks_pg,opp_prev_interceptions_pg,opp_prev_tackles_for_loss_pg,opp_prev_qb_hits_pg,opp_prev_passes_defended_pg,opp_prev_defensive_tds_pg,opp_prev_total_tackles_pg,opp_prev_redzone_att_allowed_pg,opp_prev_third_down_pct_allowed,fp_ppr
0,Joe Flacco,11252,2022,1,home,QB,20,33,7,2.0,17.0,34.5,12.91,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.5,0.0,0.75,129.9,72.85,80.75,13.5,0.0,43.5,21.0,1.5,169.0,1.0,11.6,23.06,363.41,278.94,84.47,20.00,36.53,23.35,22.24,1.82,0.76,2.76,1704.94,0.88,0.82,0.35,2.00,0.53,4.47,5.65,4.24,0.06,61.88,2.94,34.83,14.36
1,Joe Flacco,11252,2022,2,away,QB,20,5,7,2.0,17.0,34.5,12.91,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.5,0.0,0.75,129.9,72.85,80.75,13.5,0.0,43.5,21.0,1.5,169.0,1.0,11.6,21.82,311.53,202.29,109.24,19.24,33.88,21.59,25.71,1.71,0.76,2.53,1751.88,1.12,1.00,0.35,2.53,0.76,4.47,6.00,4.65,0.12,64.82,3.06,39.27,28.88
2,Joe Flacco,11252,2022,3,home,QB,20,4,7,2.0,17.0,34.5,12.91,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.5,0.0,0.75,129.9,72.85,80.75,13.5,0.0,43.5,21.0,1.5,169.0,1.0,11.6,22.12,350.82,248.35,102.47,19.94,36.82,24.71,23.94,1.53,0.88,2.47,1805.82,1.24,1.18,0.47,2.47,0.76,4.71,5.82,3.71,0.12,66.53,3.12,41.63,7.40
3,Joe Flacco,11252,2022,14,away,QB,20,2,77,2.0,17.0,34.5,12.91,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.5,0.0,0.75,129.9,72.85,80.75,13.5,0.0,43.5,21.0,1.5,169.0,1.0,11.6,17.00,272.76,163.00,109.76,16.76,31.18,17.47,26.00,0.71,1.12,1.88,1713.82,1.76,1.65,0.65,2.47,1.12,5.41,5.35,4.71,0.06,59.94,2.65,30.84,0.04
4,Joe Flacco,11252,2022,18,away,QB,20,15,28,2.0,17.0,34.5,12.91,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.5,0.0,0.75,129.9,72.85,80.75,13.5,0.0,43.5,21.0,1.5,169.0,1.0,11.6,21.94,337.53,227.71,109.82,19.82,35.76,21.94,25.00,1.35,0.94,2.29,1804.35,1.53,1.53,0.71,2.82,0.82,4.29,7.53,5.35,0.35,64.82,3.35,41.13,5.96


In [ ]:
# --- Split: within 2025, by week ---
# 2022-2024 only contain players still active in 2025, so their mean fp_ppr (~12)
# is far above 2025's (~6.8). Training across that boundary made the model
# over-predict by ~3 pts/game. Staying inside 2025 keeps one population.

SEASON = 2025
TRAIN_WEEKS = range(1, 12)     # 1-11
VALID_WEEKS = [12, 13]         # early stopping
TEST_WEEKS = range(14, 19)     # 14-18

d = model_df[model_df["season"] == SEASON]
train = d[d["week"].isin(TRAIN_WEEKS)]
valid = d[d["week"].isin(VALID_WEEKS)]
test = d[d["week"].isin(TEST_WEEKS)]

def make_pool(x):
    return Pool(x[FEATURE_COLS], x[TARGET], cat_features=CAT_COLS)

train_pool, valid_pool, test_pool = make_pool(train), make_pool(valid), make_pool(test)

for name, x in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:6s} {len(x):6,d} rows  weeks {x['week'].min():>2}-{x['week'].max():<2}  "
          f"mean {TARGET} {x[TARGET].mean():6.3f}")

# Most 2025 players have no 2024 row in this DB, so their prev_* features are all
# zero. That split is worth watching -- it tracks playing time, not just history.
cold = d["prev_games"] == 0
print(f"\n2025 rows with 2024 history: {(~cold).sum():,} ({(~cold).mean():.1%}), "
      f"mean {TARGET} {d.loc[~cold, TARGET].mean():.2f}")
print(f"2025 rows without           : {cold.sum():,} ({cold.mean():.1%}), "
      f"mean {TARGET} {d.loc[cold, TARGET].mean():.2f}")

In [ ]:
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=42,
    early_stopping_rounds=200,
    verbose=500,
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
print("best iteration:", model.get_best_iteration())

In [ ]:
def metrics(y, pred):
    y, pred = np.asarray(y, float), np.asarray(pred, float)
    return np.sqrt(np.mean((y - pred) ** 2)), np.mean(np.abs(y - pred))

print("=== fit ===")
for name, x, pool in [("train", train, train_pool), ("valid", valid, valid_pool), ("test", test, test_pool)]:
    r, m = metrics(x[TARGET], model.predict(pool))
    print(f"{name:6s} n={len(x):6,d}  RMSE {r:6.3f}  MAE {m:6.3f}")

# Baselines matter more than the raw RMSE: the naive "use last season's average"
# model is the bar any of this has to clear.
print(f"\n=== test (weeks {test['week'].min()}-{test['week'].max()}) vs baselines ===")
y_test = test[TARGET].to_numpy()
pred = model.predict(test_pool)
for label, p in [
    ("CatBoost", pred),
    ("prev_fp_ppr_avg passthrough", test["prev_fp_ppr_avg"].to_numpy()),
    ("constant = train mean", np.full(len(test), train[TARGET].mean())),
    ("constant = test mean (oracle)", np.full(len(test), y_test.mean())),
]:
    r, m = metrics(y_test, p)
    print(f"{label:32s} RMSE {r:6.3f}  MAE {m:6.3f}")

print(f"\ncalibration -> actual {y_test.mean():.3f} | predicted {pred.mean():.3f}")

scored = test.copy()
scored["pred"] = pred

print("\n=== by position ===")
for pos, g in scored.groupby("position_abbr", observed=True):
    r, m = metrics(g[TARGET], g["pred"])
    print(f"{pos:3s} n={len(g):5,d}  RMSE {r:6.3f}  MAE {m:6.3f}  "
          f"actual {g[TARGET].mean():5.2f}  pred {g['pred'].mean():5.2f}")

# The honest check: how much of the win comes from players who simply have no
# prior-season row (all-zero features, and reliably low scorers)?
print("\n=== by prior-season availability ===")
scored["has_prev"] = scored["prev_games"] > 0
for flag, g in scored.groupby("has_prev"):
    r, m = metrics(g[TARGET], g["pred"])
    rb, mb = metrics(g[TARGET], g["prev_fp_ppr_avg"])
    print(f"has_prev={str(flag):5s} n={len(g):5,d}  actual {g[TARGET].mean():5.2f} | "
          f"CatBoost RMSE {r:6.3f} MAE {m:6.3f} | passthrough RMSE {rb:6.3f} MAE {mb:6.3f}")

In [ ]:
imp = (
    pd.DataFrame({"feature": FEATURE_COLS,
                  "importance": model.get_feature_importance(train_pool)})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

top = imp.head(25).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(top["feature"], top["importance"])
ax.set_title(f"Top 25 features — target: {TARGET}")
fig.tight_layout()

imp.head(25)